In [0]:
%pip install xgboost==3.2.0
%pip install shap==0.51.0

In [0]:
%restart_python

In [0]:
# importing libraries

import numpy as np
import pandas as pd
import shap
import mlflow
import mlflow.xgboost
from pyspark.ml.functions import vector_to_array
from pyspark.sql.functions import col
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

import os
os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

username = spark.sql("SELECT current_user()").collect()[0][0]
mlflow.set_experiment(f"/Users/{username}/explainability")

# Feature names

APP_FEATURE_NAMES = [
    "log_income",
    "address_stability",
    "under_25",
    "name_email_similarity",
    "days_since_request",
    "zip_count_4w"
]

TXN_FEATURE_NAMES = [
    "log_amount",
    "amount_balance_ratio",
    "transaction_hour",
    "transaction_dayofweek",
    "merchant_fraud_rate",
    "customer_avg_transaction_amount",
    "merchant_category_index",
    "device_type_index",
    "channel_grouped_index",
    "location_city_grouped_index"
]

# Loading production models from mlflow registry

def load_production_model(model_name):
    """
    Load the latest version of a registered MLflow model.
    Falls back to most recent run if registry lookup fails.
    """
    try:
        # Try Unity Catalog registry first
        model = mlflow.xgboost.load_model(f"models:/{model_name}/latest")
        print(f"  ✅ Loaded {model_name} from MLflow Registry")
        return model

    except Exception as e:
        print(f"  ⚠️  Registry load failed for {model_name}, trying recent run...")

        # Finding the most recent XGB run in supervised_models experiment
        from mlflow.tracking import MlflowClient
        client     = MlflowClient()
        experiment = mlflow.get_experiment_by_name(f"/Users/{username}/supervised_models")

        # Parse dataset from model name
        dataset_lower = model_name.split("_")[-1].lower()
        run_name      = f"XGB_{dataset_lower}"

        runs = mlflow.search_runs(
            experiment_ids=[experiment.experiment_id],
            filter_string=f"tags.mlflow.runName = '{run_name}'",
            order_by=["start_time DESC"],
            max_results=1
        )

        if len(runs) == 0:
            raise RuntimeError(f"No runs found for {model_name}")

        run_id    = runs.iloc[0]["run_id"]
        model_uri = f"runs:/{run_id}/xgb_model"
        model     = mlflow.xgboost.load_model(model_uri)
        print(f"  ✅ Loaded {model_name} from run {run_id[:8]}")
        return model


print("Loading production XGBoost models from MLflow...")
xgb_app = load_production_model("Modelo_Fraude_XGB_Applications")
xgb_txn = load_production_model("Modelo_Fraude_XGB_Transactions")

# Loading test data

def load_test_data(test_table, sample_n=50000):
    """Load test set as numpy with stratified sampling for SHAP performance."""
    test_df = spark.table(test_table) \
                   .withColumn("is_fraud", col("is_fraud").cast("double"))

    # Stratified sample
    fraud_sample = test_df.filter("is_fraud = 1").limit(sample_n // 10)
    legit_sample = test_df.filter("is_fraud = 0").limit(sample_n)
    test_sample  = fraud_sample.union(legit_sample)

    arr = test_sample.withColumn("features_arr", vector_to_array("features"))
    pdf = arr.select("features_arr", "is_fraud").toPandas()
    X   = np.array(pdf["features_arr"].tolist())
    y   = pdf["is_fraud"].values
    return X, y


print("\nLoading application test data...")
X_app_test, y_app_test = load_test_data(
    "workspace.ml_layer.application_test_features", sample_n=50000
)

print("Loading transaction test data...")
X_txn_test, y_txn_test = load_test_data(
    "workspace.ml_layer.transaction_test_features", sample_n=100000
)

# Verify feature counts match
print(f"\n  App features in data:  {X_app_test.shape[1]}")
print(f"  App features in list:  {len(APP_FEATURE_NAMES)}")
print(f"  Txn features in data:  {X_txn_test.shape[1]}")
print(f"  Txn features in list:  {len(TXN_FEATURE_NAMES)}")


# SHAP Analysis

def run_shap_analysis(model, X_test, feature_names, dataset_name, n_samples=2000):
    """Compute SHAP values and generate plots for the production model."""
    
    # Self-correct feature names if length mismatch
    actual_n = X_test.shape[1]
    if actual_n != len(feature_names):
        print(f"  ⚠️  Adjusting feature_names: {len(feature_names)} → {actual_n}")
        if actual_n < len(feature_names):
            feature_names = feature_names[:actual_n]
        else:
            feature_names = feature_names + [f"feat_{i}" for i in range(len(feature_names), actual_n)]

    print(f"\n  Computing SHAP values for {dataset_name}...")

    background  = shap.sample(X_test, 100, random_state=42)
    explainer   = shap.TreeExplainer(model, background)

    idx         = np.random.choice(len(X_test), min(n_samples, len(X_test)), replace=False)
    X_sample    = X_test[idx]
    shap_values = explainer.shap_values(X_sample)

    if isinstance(shap_values, list):
        shap_values = shap_values[1]

    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      show=False, max_display=len(feature_names))
    plt.title(f"SHAP Feature Impact — {dataset_name}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"/tmp/shap_summary_{dataset_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Bar plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_sample, feature_names=feature_names,
                      plot_type="bar", show=False, max_display=len(feature_names))
    plt.title(f"SHAP Mean |Value| — {dataset_name}", fontsize=14, pad=20)
    plt.tight_layout()
    plt.savefig(f"/tmp/shap_bar_{dataset_name}.png", dpi=150, bbox_inches="tight")
    plt.show()

    # Dependence plot for top feature
    top_idx     = np.abs(shap_values).mean(axis=0).argmax()
    top_feature = feature_names[top_idx]
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(top_idx, shap_values, X_sample,
                         feature_names=feature_names, show=False)
    plt.title(f"SHAP Dependence — {top_feature} | {dataset_name}", fontsize=14)
    plt.tight_layout()
    plt.savefig(f"/tmp/shap_dependence_{dataset_name}.png",
                dpi=150, bbox_inches="tight")
    plt.show()

    # Feature importance table
    mean_shap = pd.DataFrame({
        "feature"      : feature_names,
        "mean_abs_shap": np.abs(shap_values).mean(axis=0),
        "mean_shap"    : shap_values.mean(axis=0)
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    mean_shap["rank"] = range(1, len(mean_shap) + 1)

    print(f"\n  SHAP Feature Ranking — {dataset_name}")
    print(mean_shap.to_string(index=False))

    return shap_values, mean_shap, explainer, idx


# Running SHAP for both datasets

with mlflow.start_run(run_name="SHAP_applications"):
    shap_vals_app, shap_importance_app, explainer_app, idx_app = run_shap_analysis(
        xgb_app, X_app_test, APP_FEATURE_NAMES, "applications"
    )
    mlflow.log_dict(
        shap_importance_app.to_dict(orient="records"),
        "shap_importance_applications.json"
    )

with mlflow.start_run(run_name="SHAP_transactions"):
    shap_vals_txn, shap_importance_txn, explainer_txn, idx_txn = run_shap_analysis(
        xgb_txn, X_txn_test, TXN_FEATURE_NAMES, "transactions"
    )
    mlflow.log_dict(
        shap_importance_txn.to_dict(orient="records"),
        "shap_importance_transactions.json"
    )

# XGBOOST Feature Importance

def plot_xgb_feature_importance(model, feature_names, dataset_name):
    importance_types = ["gain", "weight", "cover"]
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    # Handle feature count mismatch
    if X_app_test.shape[1] != len(feature_names) and "application" in dataset_name.lower():
        feature_names = feature_names[:X_app_test.shape[1]]

    for ax, imp_type in zip(axes, importance_types):
        scores = model.get_booster().get_score(importance_type=imp_type)
        named_scores = {
            feature_names[int(k.replace("f", ""))]: v
            for k, v in scores.items()
            if k.replace("f", "").isdigit()
            and int(k.replace("f", "")) < len(feature_names)
        }
        sorted_scores = dict(sorted(named_scores.items(),
                                    key=lambda x: x[1], reverse=True))

        ax.barh(list(sorted_scores.keys()), list(sorted_scores.values()),
                color="#2563EB", alpha=0.8)
        ax.set_title(f"Importance by {imp_type.upper()}", fontsize=12)
        ax.set_xlabel(imp_type)
        ax.invert_yaxis()
        ax.grid(axis="x", alpha=0.3)

    fig.suptitle(f"XGBoost Feature Importance — {dataset_name}",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"/tmp/xgb_importance_{dataset_name}.png",
                dpi=150, bbox_inches="tight")
    plt.show()
    return named_scores


importance_app = plot_xgb_feature_importance(xgb_app, APP_FEATURE_NAMES, "applications")
importance_txn = plot_xgb_feature_importance(xgb_txn, TXN_FEATURE_NAMES, "transactions")

# Reason Codes

def generate_reason_codes(model, explainer, X, feature_names,
                          dataset_name, n_examples=10):
    """Generate human-readable reason codes from production model SHAP values."""
    
    if X.shape[1] != len(feature_names):
        feature_names = feature_names[:X.shape[1]]
    
    shap_vals = explainer.shap_values(X[:n_examples])
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    y_proba = model.predict_proba(X[:n_examples])[:, 1]

    records = []
    for i in range(n_examples):
        fraud_score = y_proba[i]
        sv          = shap_vals[i]

        feature_impacts = sorted(
            zip(feature_names, sv, X[i]),
            key=lambda x: abs(x[1]),
            reverse=True
        )

        reasons = []
        for feat_name, shap_val, feat_val in feature_impacts[:3]:
            direction = "↑ increases" if shap_val > 0 else "↓ decreases"
            reasons.append(
                f"{feat_name} ({feat_val:.3f}) {direction} fraud risk "
                f"[SHAP: {shap_val:+.4f}]"
            )

        if fraud_score >= 0.80:
            risk_tier = "🔴 HIGH RISK"
        elif fraud_score >= 0.40:
            risk_tier = "🟡 MEDIUM RISK"
        else:
            risk_tier = "🟢 LOW RISK"

        records.append({
            "example_id"  : i,
            "fraud_score" : round(fraud_score, 4),
            "risk_tier"   : risk_tier,
            "reason_1"    : reasons[0] if len(reasons) > 0 else "",
            "reason_2"    : reasons[1] if len(reasons) > 1 else "",
            "reason_3"    : reasons[2] if len(reasons) > 2 else "",
        })

    return pd.DataFrame(records)


print("\n" + "="*70)
print("  REASON CODES — Applications (sample of 10)")
print("="*70)
reason_codes_app = generate_reason_codes(
    xgb_app, explainer_app, X_app_test, APP_FEATURE_NAMES,
    "applications", n_examples=10
)
display(reason_codes_app)

print("\n" + "="*70)
print("  REASON CODES — Transactions (sample of 10)")
print("="*70)
reason_codes_txn = generate_reason_codes(
    xgb_txn, explainer_txn, X_txn_test, TXN_FEATURE_NAMES,
    "transactions", n_examples=10
)
display(reason_codes_txn)

# Save reason codes to Delta
spark.createDataFrame(reason_codes_app).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.reason_codes_applications")

spark.createDataFrame(reason_codes_txn).write.format("delta") \
    .mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.ml_layer.reason_codes_transactions")

print("\n✅ Reason codes saved to Delta tables")

# Waterfall plots for individual predictions

def explain_single_prediction(model, explainer, X, feature_names,
                               dataset_name, example_idx=0):
    """Waterfall plot for one specific prediction."""
    
    if X.shape[1] != len(feature_names):
        feature_names = feature_names[:X.shape[1]]
    
    x_single  = X[example_idx:example_idx+1]
    shap_vals = explainer.shap_values(x_single)
    if isinstance(shap_vals, list):
        shap_vals = shap_vals[1]

    fraud_score = model.predict_proba(x_single)[0, 1]
    print(f"\n  Explaining prediction {example_idx} | "
          f"Fraud score: {fraud_score:.4f} | "
          f"{'🔴 FRAUD' if fraud_score > 0.5 else '🟢 LEGIT'}")

    base_value = explainer.expected_value
    if isinstance(base_value, list):
        base_value = base_value[1]
    elif hasattr(base_value, '__len__') and len(base_value) > 1:
        base_value = base_value[1]

    shap.waterfall_plot(
        shap.Explanation(
            values        = shap_vals[0],
            base_values   = base_value,
            data          = x_single[0],
            feature_names = feature_names
        ),
        show=False
    )
    plt.title(f"Prediction Explanation — {dataset_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"/tmp/waterfall_{dataset_name}_{example_idx}.png",
                dpi=150, bbox_inches="tight")
    plt.show()


# Find and explain top-scoring fraud cases
fraud_indices_app = np.where(xgb_app.predict_proba(X_app_test)[:, 1] > 0.7)[0]
if len(fraud_indices_app) > 0:
    explain_single_prediction(
        xgb_app, explainer_app, X_app_test, APP_FEATURE_NAMES,
        "applications", example_idx=fraud_indices_app[0]
    )

fraud_indices_txn = np.where(xgb_txn.predict_proba(X_txn_test)[:, 1] > 0.7)[0]
if len(fraud_indices_txn) > 0:
    explain_single_prediction(
        xgb_txn, explainer_txn, X_txn_test, TXN_FEATURE_NAMES,
        "transactions", example_idx=fraud_indices_txn[0]
    )

print("\n✅ Explainability analysis complete — using production XGBoost models")